<a href="https://colab.research.google.com/github/zloyaloha/ML/blob/main/Decision_Trees_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn 
import plotly.express as px

# Домашняя работа: деревья решений

В этой домашней работе вам предстоит научиться предсказывать цены товаров из маркетплейса Azamon.

Требования к домашней работе:
- Во всех графиках должны быть подписи через title, legend, etc.
- Во время обучения моделей проверяйте, что у вас не текут данные. Обычно это позитивно влияет на качество модели на тесте, но негативно влияет на оценку 🌚
- Если вы сдаете работу в Google Colaboratory, убедитесь, что ваша тетрадка доступна по ссылке.
- Использование мемов допускается, но необходимо соблюдать меру. Несодержательная работа, состоящая только из мемов, получает 0 баллов.

# Загрузка и подготовка данных

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving amazon_co-ecommerce_sample.csv to amazon_co-ecommerce_sample.csv


In [ ]:
dfM = pd.read_csv('amazon_co-ecommerce_sample.csv').drop(columns=[
    'product_name',
    'index',
    'uniq_id',
    'customers_who_bought_this_item_also_bought',
    'items_customers_buy_after_viewing_this_item',
    'sellers',
    'description', # text
    'product_information', # text
    'product_description', # text
    'customer_questions_and_answers', # text
    'customer_reviews', # text
    'amazon_category_and_sub_category',
])
df = dfM.copy()

## Очистка данных (1 балл)

Посмотрите на признаки. Есть ли в них пропуски? Какое соотношение между NaN'ами и общим количеством данных? Есть ли смысл выкидывать какие-либо данные из этого датасета?

Пропуски есть. Данные лучше не выкидывать.

In [ ]:
[print(df[col].value_counts()) for col in df.head(1)]
df.head()

LEGO              171
Disney            167
Oxford Diecast    156
Playmobil         147
Star Wars         120
                 ... 
Seben-Racing        1
Omnibus             1
Bentley             1
Speed Cycles        1
Super Heroes        1
Name: manufacturer, Length: 2651, dtype: int64
£9.99      189
£4.99      140
£14.99     132
£5.99      126
£6.99      126
          ... 
£41.00       1
£124.95      1
£82.02       1
£37.73       1
£21.20       1
Name: price, Length: 2625, dtype: int64
2 new     1337
3 new      981
4 new      753
5 new      590
6 new      475
          ... 
74 new       1
65 new       1
92 new       1
86 new       1
9 used       1
Name: number_available_in_stock, Length: 89, dtype: int64
1      4315
2      1427
3       768
4       524
5       351
       ... 
124       1
204       1
141       1
160       1
241       1
Name: number_of_reviews, Length: 194, dtype: int64
1.0     6435
2.0     1469
3.0      566
4.0      235
5.0      161
6.0       82
11.0      79
9.0      

,manufacturer,price,number_available_in_stock,number_of_reviews,number_of_answered_questions,average_review_rating
0,Hornby,£3.42,5 new,15,1.0,4.9 out of 5 stars
1,FunkyBuys,£16.99,NaN,2,1.0,4.5 out of 5 stars
2,ccf,£9.99,2 new,17,2.0,3.9 out of 5 stars
3,Hornby,£39.99,NaN,1,2.0,5.0 out of 5 stars
4,Hornby,£32.19,NaN,3,2.0,4.7 out of 5 stars


In [ ]:
df.isnull().sum()

manufacturer                       7
price                           1435
number_available_in_stock       2500
number_of_reviews                 18
number_of_answered_questions     765
average_review_rating             18
dtype: int64

In [ ]:
df.isnull().sum().sum() / df.count().sum()

## Подготовка данных (3 балла)

Обработайте признаки. Выполните кодирование категориальных признаков, заполните пропуски в числовых признаках. Обратите внимание, что в датасете есть признак, который разбивается на несколько подпризнаков. Что это за признак? Закодируйте и его.

Дополнительные вопросы (+ 1 балл):
- Какие из признаков в этом датасете лучше кодировать через ordinal encoding?
- Какие из признаков допустимо кодировать через one-hot?

Прим.: суммарно за эту секцию можно получить до 4 баллов.

In [ ]:
!pip install category_encoders

In [ ]:
import category_encoders as ce
encoder = ce.OrdinalEncoder(cols=['price',
 'number_available_in_stock',
 'number_of_reviews',
 'number_of_answered_questions',
 'average_review_rating'])
df = encoder.fit_transform(df)
df.head()

,manufacturer,price,number_available_in_stock,number_of_reviews,number_of_answered_questions,average_review_rating
0,Hornby,1,1,1,1,1
1,FunkyBuys,2,2,2,1,2
2,ccf,3,3,3,2,3
3,Hornby,4,2,4,2,4
4,Hornby,5,2,5,2,5


In [ ]:
df['price'] = df['price'].fillna(df['price'].mean())
df['number_available_in_stock'] = df['number_available_in_stock'].fillna(df['number_available_in_stock'].mean())
df['number_of_reviews'] = df['number_of_reviews'].fillna(df['number_of_reviews'].mean())
df['number_of_answered_questions'] = df['number_of_answered_questions'].fillna(df['number_of_answered_questions'].mean())
df['average_review_rating'] = df['average_review_rating'].fillna(df['average_review_rating'].mean())
df.isnull().sum()

manufacturer                    7
price                           0
number_available_in_stock       0
number_of_reviews               0
number_of_answered_questions    0
average_review_rating           0
dtype: int64

In [ ]:
from sklearn. preprocessing import OneHotEncoder

for i in df["amazon_category_and_sub_category"]:
  if (type(i) == float):
    continue
  category = i.split(" > ")
  for j in category:
    if j not in list(df.head(0)):
      df[j] = 0
      id = df.index[df["amazon_category_and_sub_category"] == i ].tolist()
      for k in id:
        df[j][k] = 1

In [ ]:
df.head()

In [ ]:
y = df["price"].T
X = df.drop(columns = ["price"], axis = 1)
X.head()

,manufacturer,number_available_in_stock,number_of_reviews,number_of_answered_questions,average_review_rating
0,Hornby,1,1,1,1
1,FunkyBuys,2,2,1,2
2,ccf,3,3,2,3
3,Hornby,2,4,2,4
4,Hornby,2,5,2,5


In [ ]:
y.head()

0    1
1    2
2    3
3    4
4    5
Name: price, dtype: int64

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn. preprocessing import OneHotEncoder

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.65, random_state=42)
encoder = OneHotEncoder(handle_unknown='ignore')
type_encoded_train = encoder.fit_transform(X_train['manufacturer'].to_numpy().reshape(-1, 1)).todense()
type_encoded_test = encoder.transform(X_test['manufacturer'].to_numpy().reshape(-1, 1)).todense()
feature_names = encoder.get_feature_names_out()
type_encoded_train_df = pd.DataFrame(data=type_encoded_train, columns=feature_names)
type_encoded_test_df = pd.DataFrame(data=type_encoded_test, columns=feature_names)

type_encoded_train_df.index = X_train.index
type_encoded_test_df.index = X_test.index

X_train = pd.concat([X_train.drop(columns=['manufacturer']), type_encoded_train_df], axis=1)
X_test = pd.concat([X_test.drop(columns=['manufacturer']), type_encoded_test_df], axis=1)

In [ ]:
X_train = X_train.drop(columns = ['amazon_category_and_sub_category'],axis = 1)
X_test = X_test.drop(columns = ['amazon_category_and_sub_category'],axis = 1)

# Обучение модели (3 балла)

## Бейзлайн

Обучите базовую модель. Для этого используйте `sklearn.dummy.DummyRegressor`. Какое качество она показывает на тесте? Посчитайте MSE, RMSE.

In [ ]:
X_train.head()

In [ ]:
y_train.head()

4792     231
8854     171
6250      34
5936    1931
425      264
Name: price, dtype: int64

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

dummy_regr = DummyRegressor()
dummy_regr.fit(X_train, y_train)
y_pred = dummy_regr.predict(X_test)
print(type(y_pred), type(y_test))
print(mean_squared_error(y_pred, y_test))
print(mean_squared_error(y_pred, y_test, squared=False))

<class 'numpy.ndarray'> <class 'pandas.core.series.Series'>
482566.15090560605
694.6698143043254


## Дерево решений

Обучите регрессионное дерево решений, проверьте качество этой модели на тестовой выборке. Улучшилось ли качество по сравнению с базовой моделью? Оцените r2_score обученной модели.

In [ ]:
from sklearn import tree
from sklearn.metrics import r2_score
clf = tree.DecisionTreeRegressor()
clf = clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
print(r2_score(y_pred, y_test))

-1.2650035636034342


## Линейная регрессия

Попробуйте обучить линейную регрессию с параметрами по умолчанию. Оцените r2_score на тестовой выборке. Сравните качество с деревом решений. 

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression
reg = LinearRegression().fit(X_train, y_train)
y_pred = reg.predict(X_test)
print(r2_score(y_pred, y_test))

-0.246882793079926


# Гиперпараметры (2 балла)

Переберите несколько гиперпараметров (не более двух-трёх). Обратите внимание, как эти параметры влияют на ошибку модели на тестовой выборке. Постройте для глубины дерева график переобучения (fitting curve) аналогичный тому, что мы строили на занятии. Найдите глубину дерева, начиная с которой модель начинает переобучаться.

# Простое ансамблирование (1 балл)

В этой секции мы реализуем простой ансамбль деревьев.

In [ ]:
class EnsembleTreeRegressor:
    def __init__(self, num_trees=5, samples_frac=0.8, **model_kwargs):
        self.num_trees= num_trees
        self._samples_frac = 0.8
        self._trees = [DecisionTreeRegressor(**model_kwargs) for _ in range(num_trees)]
    def fit(self, x, y: pd.Series):
        x = pd.DataFrame(x)
        y = y.reset_index(drop=True)
        for tree in self._trees:
            tree_x = x.sample(frac=self._samples_frac, random_state=42)
            tree_y = y[tree_x.index]
            tree.fit(tree_x, tree_y)
        return self

    def predict(self, x: pd.DataFrame):
        x = pd.DataFrame(x)
        res = []
        for i in range(self.num_trees):
          res.append(self._trees[i].predict(x))
        return sum(res) / len(res)

Проверьте, работает ли этот ансамбль лучше обычного дерева с параметрами по умолчанию?

Дополнительно переберите максимальную глубину дерева. Проверьте, насколько отличается момент начала переобучения у одиночного дерева и у ансамбля. Зависит ли этот момент от числа деревьев (`num_trees`)? От числа примеров для каждого дерева (`samples_frac`)? Постройте график fitting curve.